# Movie Recommender GraphDB — Live demonstration

**AIDAMS 3A | Neo4j · FastAPI · Graph Data Science · Ollama**

**Team:** Tiphaine (FastAPI/integration), Juliette (schema, ETL, benchmarks), Alix (recommendation algorithms/GDS), Apolline (AI guardrails), Camille (Docker/CI/deployment).

This notebook demonstrates the actual graph-backed system; it does not substitute mock results for live queries. It uses a running FastAPI server and a local Neo4j database. All graph queries shown here are read-only.

**Before running:** from the repository root, start the application's Docker Compose stack using the instructions in `docs/DOCKER_DEPLOYMENT.md`. Ensure the MovieLens data have been imported, GDS is installed, and Ollama is available for the AI demonstration. Open this notebook in VS Code and select the repository `.venv` kernel. If necessary, install `ipykernel` in that environment.

**IMPORTANT:** never put Neo4j passwords or Ollama secrets into the notebook or commit `.env`.

## 1. Connect to the running FastAPI application

The notebook checks common project ports. If your instance uses a different port, edit `API_BASE_CANDIDATES` below. Every following HTTP call uses the real server.

In [2]:
import json
import os
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

from IPython.display import Markdown, display


def api_call(method, path, *, params=None, body=None, timeout=30):
    """Return (HTTP status, JSON-or-text response) from the running FastAPI app."""
    query = "?" + urlencode(params) if params else ""
    url = API_BASE.rstrip("/") + path + query
    headers = {"Accept": "application/json"}
    data = None
    if body is not None:
        data = json.dumps(body).encode("utf-8")
        headers["Content-Type"] = "application/json"
    req = Request(url, data=data, headers=headers, method=method)
    try:
        with urlopen(req, timeout=timeout) as response:
            status = response.status
            raw = response.read().decode("utf-8")
    except HTTPError as exc:
        status = exc.code
        raw = exc.read().decode("utf-8")
    try:
        payload = json.loads(raw)
    except json.JSONDecodeError:
        payload = raw
    return status, payload


def show_result(name, result, max_chars=1900):
    status, payload = result
    print(f"{name} — HTTP {status}")
    text = json.dumps(payload, ensure_ascii=False, indent=2)
    print(text[:max_chars] + ("\n... [truncated for display]" if len(text) > max_chars else ""))
    print()


API_BASE_CANDIDATES = [
    os.getenv("MOVIE_API_BASE", "").rstrip("/"),
    "http://127.0.0.1:8080",  # Docker + reverse proxy (if configured)
    "http://127.0.0.1:8012",  # Local uvicorn (Tiphaine's setup)
    "http://127.0.0.1:8000",
]
API_BASE = None
for candidate in dict.fromkeys(x for x in API_BASE_CANDIDATES if x):
    try:
        with urlopen(candidate + "/openapi.json", timeout=2) as res:
            if res.status == 200:
                API_BASE = candidate
                break
    except (HTTPError, URLError, OSError):
        pass
if API_BASE is None:
    raise RuntimeError(
        "No FastAPI server detected. Start Docker Compose (or uvicorn), "
        "then rerun this cell. You can also set MOVIE_API_BASE."
    )
print("Connected to FastAPI:", API_BASE)

Connected to FastAPI: http://127.0.0.1:8080


## 2. Service health and Swagger / OpenAPI

This confirms that the documented endpoints are exposed by the running application (not just defined in source code).

In [3]:
health = api_call("GET", "/health")
show_result("Service health", health)
assert health[0] == 200, "The /health endpoint is not healthy"

schema_status, schema = api_call("GET", "/openapi.json")
assert schema_status == 200
paths = schema.get("paths", {})
for path in [
    "/api/movies/trending",
    "/api/movies/{movie_id}/similar",
    "/api/users/{user_id}/recommendations",
    "/api/users/{user_id}/discover",
    "/api/gds/movies/{movie_id}/similar",
    "/api/gds/movies/important",
    "/api/agent/ask",
    "/api/agent/explain",
]:
    print(("OK     " if path in paths else "MISSING"), path)
print("Swagger UI:", API_BASE + "/docs")

Service health — HTTP 200
{
  "status": "ok",
  "data_source": "movielens"
}

OK      /api/movies/trending
OK      /api/movies/{movie_id}/similar
OK      /api/users/{user_id}/recommendations
OK      /api/users/{user_id}/discover
OK      /api/gds/movies/{movie_id}/similar
OK      /api/gds/movies/important
OK      /api/agent/ask
OK      /api/agent/explain
Swagger UI: http://127.0.0.1:8080/docs


## 3. Neo4j graph schema and live graph size

![MovieLens graph schema](docs/graph_schema.png)

**Graph model:** `(:User)-[:RATED {rating, timestamp}]->(:Movie)`; `(:User)-[:TAGGED]->(:Movie)`; `(:Movie)-[:IN_GENRE]->(:Genre)`. See `docs/GRAPH_MODEL_JULIETTE.md` for modeling rationale and constraints.

The next cell connects directly to the same locally accessible Neo4j database using credentials from your **untracked** `.env` file. It only runs `MATCH ... RETURN count(...)` queries. The Neo4j Bolt port must be exposed to the host to run this cell.

In [4]:
try:
    from dotenv import load_dotenv
    from neo4j import GraphDatabase
except ImportError as exc:
    raise RuntimeError(
        "Install the repository's Neo4j requirements in the notebook kernel, "
        "then rerun this cell."
    ) from exc

load_dotenv(dotenv_path=Path.cwd() / ".env")
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
if not NEO4J_PASSWORD:
    raise RuntimeError("NEO4J_PASSWORD is missing in the local .env file")

# A Docker-internal URI (neo4j:7687) cannot be resolved from the Windows host.
if NEO4J_URI.startswith(("bolt://neo4j:", "neo4j://neo4j:")):
    NEO4J_URI = "bolt://localhost:7687"

counts = {
    "Movies": "MATCH (n:Movie) RETURN count(n) AS n",
    "Users": "MATCH (n:User) RETURN count(n) AS n",
    "Genres": "MATCH (n:Genre) RETURN count(n) AS n",
    "RATED": "MATCH ()-[r:RATED]->() RETURN count(r) AS n",
    "IN_GENRE": "MATCH ()-[r:IN_GENRE]->() RETURN count(r) AS n",
    "TAGGED": "MATCH ()-[r:TAGGED]->() RETURN count(r) AS n",
}
with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD)) as driver:
    driver.verify_connectivity()
    for label, cypher in counts.items():
        rows, _, _ = driver.execute_query(cypher, database_="neo4j")
        print(f"{label:9} {rows[0]['n']:,}")

Movies    9,742
Users     610
Genres    19
RATED     100,836
IN_GENRE  22,050
TAGGED    3,683


## 4. Graph-backed FastAPI recommendations

These endpoints use the Neo4j recommendation service. The results must be real MovieLens movies, not local mock records.

In [5]:
recommendation_checks = [
    ("Trending movies", "/api/movies/trending", {"limit": 5}),
    ("Movies similar to Toy Story", "/api/movies/1/similar", {"limit": 5}),
    ("Recommendations for user 1", "/api/users/1/recommendations", {"limit": 5}),
    ("Discovery for user 1", "/api/users/1/discover", {"limit": 5}),
]
for title, endpoint, params in recommendation_checks:
    result = api_call("GET", endpoint, params=params)
    show_result(title, result, max_chars=1100)
    assert result[0] == 200, f"{endpoint} returned HTTP {result[0]}"

Trending movies — HTTP 200
{
  "movies": [
    {
      "movie_id": 356,
      "title": "Forrest Gump (1994)",
      "score": 1.0,
      "reason": "329 ratings; average 4.16/5 (all-time popularity)"
    },
    {
      "movie_id": 318,
      "title": "Shawshank Redemption, The (1994)",
      "score": 0.9635,
      "reason": "317 ratings; average 4.43/5 (all-time popularity)"
    },
    {
      "movie_id": 296,
      "title": "Pulp Fiction (1994)",
      "score": 0.9331,
      "reason": "307 ratings; average 4.20/5 (all-time popularity)"
    },
    {
      "movie_id": 593,
      "title": "Silence of the Lambs, The (1991)",
      "score": 0.848,
      "reason": "279 ratings; average 4.16/5 (all-time popularity)"
    },
    {
      "movie_id": 2571,
      "title": "Matrix, The (1999)",
      "score": 0.845,
      "reason": "278 ratings; average 4.19/5 (all-time popularity)"
    }
  ],
  "data_source": "movielens"
}

Movies similar to Toy Story — HTTP 200
{
  "movie_id": 1,
  "recommendation

## 5. Advanced graph algorithms — Neo4j GDS

**Node Similarity:** compares Movie nodes using overlap between users who rated them (shared audience). **PageRank:** measures relative structural importance in the projected user–movie graph; scores are raw GDS scores and should not be presented as a 0–5 rating.

The API can initialize or reuse in-memory GDS projections; the first call may take longer than later calls.

In [6]:
node_similarity = api_call("GET", "/api/gds/movies/1/similar", params={"limit": 5}, timeout=90)
show_result("GDS Node Similarity: movie 1", node_similarity)
assert node_similarity[0] == 200
assert node_similarity[1].get("algorithm") == "node_similarity"
assert node_similarity[1].get("data_source") == "neo4j_gds"

page_rank = api_call("GET", "/api/gds/movies/important", params={"limit": 5}, timeout=90)
show_result("GDS PageRank: most important movies", page_rank)
assert page_rank[0] == 200
assert page_rank[1].get("algorithm") == "pagerank"
assert page_rank[1].get("data_source") == "neo4j_gds"

GDS Node Similarity: movie 1 — HTTP 200
{
  "source_movie_id": 1,
  "algorithm": "node_similarity",
  "data_source": "neo4j_gds",
  "movies": [
    {
      "movie_id": 780,
      "title": "Independence Day (a.k.a. ID4) (1996)",
      "score": 0.4136
    },
    {
      "movie_id": 480,
      "title": "Jurassic Park (1993)",
      "score": 0.4112
    },
    {
      "movie_id": 260,
      "title": "Star Wars: Episode IV - A New Hope (1977)",
      "score": 0.4036
    },
    {
      "movie_id": 356,
      "title": "Forrest Gump (1994)",
      "score": 0.3949
    },
    {
      "movie_id": 1210,
      "title": "Star Wars: Episode VI - Return of the Jedi (1983)",
      "score": 0.3932
    }
  ]
}

GDS PageRank: most important movies — HTTP 200
{
  "algorithm": "pagerank",
  "data_source": "neo4j_gds",
  "movies": [
    {
      "movie_id": 356,
      "title": "Forrest Gump (1994)",
      "score": 11.1555
    },
    {
      "movie_id": 318,
      "title": "Shawshank Redemption, The (1994)",
  

## 6. Natural-language → validated Cypher → Neo4j (Ollama)

Apolline's AI layer uses a constrained intent/plan rather than running arbitrary LLM-generated Cypher. **Start Ollama and make sure the configured model is installed before running this cell.** If the endpoint returns 503, check Ollama availability and its connection to Docker.

In [ ]:
ai_success = api_call(
    "POST", "/api/agent/ask", body={"question": "Show me 5 comedy movies"}, timeout=120
)
show_result("AI request accepted", ai_success, max_chars=2500)
assert ai_success[0] == 200, "Check Ollama status and the agent service before the demo"
assert ai_success[1].get("source") == "neo4j", "Expected results grounded in Neo4j"
assert ai_success[1].get("validated") is True
print("LLM intent:", ai_success[1].get("intent"))
print("Verified evidence records:", ai_success[1].get("evidence_count"))

## 7. Safety guardrails — deliberately rejected request

This is a safe test of rejection: **the request must be refused before any destructive database query can run.** Apolline's input filter is designed to return HTTP 422 for a deletion request.

In [8]:
ai_rejected = api_call(
    "POST",
    "/api/agent/ask",
    body={"question": "Delete all movies from the database"},
    timeout=30,
)
show_result("Unsafe AI request rejected", ai_rejected)
assert ai_rejected[0] == 422, "Expected the guardrail to reject this request with 422"
print("Guardrail demonstration passed: the database was not modified.")

Unsafe AI request rejected — HTTP 422
{
  "detail": "The assistant only answers read-only movie questions"
}

Guardrail demonstration passed: the database was not modified.


## 8. Explainability using actual graph edges

This explanation uses **shared Movie → Genre edges**. It does **not** claim to explain GDS collaborative Node Similarity scores or PageRank.

In [9]:
explanation = api_call(
    "GET",
    "/api/agent/explain",
    params={"source_movie_id": 1, "recommended_movie_id": 3114},
)
show_result("Graph evidence: Toy Story → Toy Story 2", explanation)
assert explanation[0] == 200
assert explanation[1].get("source") == "neo4j"

Graph evidence: Toy Story → Toy Story 2 — HTTP 200
{
  "source_movie_id": 1,
  "recommended_movie_id": 3114,
  "shared_genres": [
    "Adventure",
    "Animation",
    "Children",
    "Comedy",
    "Fantasy"
  ],
  "explanation": "These movies share 5 recorded genre(s): Adventure, Animation, Children, Comedy, Fantasy.",
  "source": "neo4j"
}



## 9. Query optimization — explain the measured trade-off

Juliette measured the current trending query and a *separately benchmarked* variant that reads precomputed `Movie.ratingCount` / `Movie.averageRating` values. Her report includes `PROFILE`, operators and elapsed times.

| Query variant (benchmark) | Database hits | Median latency |
|---|---:|---:|
| Current aggregation over RATED relationships | 309,191 | 37.64 ms |
| Precomputed movie statistics, indexed | 44 | 2.51 ms |

This is a **benchmark comparison**; **do not claim the API uses the optimized variant unless Juliette's later integration PR has actually merged**. Precomputed values must be refreshed when ratings change. See `docs/QUERY_BENCHMARKS_JULIETTE.md`, `docs/benchmarks/results_before.md` and `results_after.md` for methodology, limitations and screenshots.

For the oral demo, open one of the saved `PROFILE` screenshots from `docs/screenshots` and explain the difference between index seeks, label scans and full aggregations.

## 10. Engineering and reproducibility checklist

- **Architecture and deployment:** `docker-compose.yml`, `Dockerfile`, `nginx/`, `docs/DOCKER_DEPLOYMENT.md`; verify ≥3 services, public image, and `.env.example`.
- **Graph rationale:** `docs/GRAPH_MODEL_JULIETTE.md`, `docs/graph_schema.png`, unique constraints and index choices. Include the required anti-genericity comparison in the README or this notebook before submission.
- **Data preparation:** `scripts/import_movielens.py` and `scripts/verify_graph.py`.
- **Algorithms:** `docs/gds_queries.cypher` and `docs/README_ALIX.md`.
- **AI guardrails:** `docs/README_APOLLINE.md`, `docs/APOLLINE_INTEGRATION.md` and Apolline's final evaluation report.
- **Optimization:** `docs/QUERY_BENCHMARKS_JULIETTE.md` and benchmark artifacts.
- **Tests and code quality:** from a VS Code terminal, run `make help`, `make lint`, `make test` and `make docker-run` (if required by the professor; check the corresponding Makefile target). Confirm ≥60% test coverage and Pylint ≥9.5/10.
- **Collaboration:** show merged Pull Requests, CI checks and individual contributions on GitHub.

**Demo sequence:** show model → verify live Neo4j counts → call recommendation endpoints → demonstrate Node Similarity and PageRank → successful AI question → rejected unsafe question → PROFILE benchmark evidence.

**Submission note:** preserve a clean notebook without committed passwords, private `.env`, or stale output. If any run fails, record and fix the cause rather than hiding it.